BatchNorm katmanını gizli katmandan sonra ekle. Eğitim sırasında batch istatistiği, tahmin sırasında running mean kullan. BatchNorm'lu ve BatchNorm'suz modelin dev loss'unu karşılaştır.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline


In [2]:

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
block_size = 3

In [3]:


def build_dataset(words):  
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


Dikkat: $b_1$ bias parametresi kullanılmaz (`#b1 = ...`), çünkü BatchNorm'daki ortalama çıkarma işlemi sabiti yok eder!

In [4]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
#b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.randn(vocab_size,                      generator=g) * 0

# BatchNorm parameters
bngain = torch.ones((1, n_hidden)) # std yi 1 ile başlatmak için ones
bnbias = torch.zeros((1, n_hidden)) # ortalamayı baslangıcta sıfır tutmak için zeros

bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden)) 

parameters = [C, W1, W2, b2, bngain, bnbias]
print("Toplam parametre:", sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

Toplam parametre: 12097


### batch norm model eğitimi

In [ ]:
max_steps = 30000
batch_size = 32
lossi = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]
  
  # forward pass
  emb = C[Xb] 
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 
  
  # BatchNorm layer

  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
  
  with torch.no_grad():
    # tek bir girdinin ortalamsı ve standart sapması hesaplanamayacağı için tüm batchler boyunca birikimli olarak ilerleyecek
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi


  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Yb)
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 20000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  if i % 10000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

      0/  30000: 3.3239
  10000/  30000: 2.0322
  20000/  30000: 2.5675


### Inference: `running_mean` ve `running_std` Kullanımı 

In [6]:
@torch.no_grad()
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.146427869796753
val 2.1610209941864014


### BatchNormsuz Model ile Karşılaştırma

In [7]:
# BatchNorm'suz model
g_nobn = torch.Generator().manual_seed(2147483647)
C_nobn  = torch.randn((vocab_size, n_embd), generator=g_nobn)
W1_nobn = torch.randn((n_embd * block_size, n_hidden), generator=g_nobn) * (5/3)/((n_embd * block_size)**0.5)
b1_nobn = torch.randn(n_hidden, generator=g_nobn) * 0.01
W2_nobn = torch.randn((n_hidden, vocab_size), generator=g_nobn) * 0.01
b2_nobn = torch.randn(vocab_size, generator=g_nobn) * 0
params_nobn = [C_nobn, W1_nobn, b1_nobn, W2_nobn, b2_nobn]
for p in params_nobn: p.requires_grad = True

for i in range(30000):
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g_nobn)
  emb = C_nobn[Xtr[ix]]
  h = torch.tanh(emb.view(emb.shape[0], -1) @ W1_nobn + b1_nobn)
  logits = h @ W2_nobn + b2_nobn
  loss = F.cross_entropy(logits, Ytr[ix])
  for p in params_nobn: p.grad = None
  loss.backward()
  lr = 0.1 if i < 20000 else 0.01
  for p in params_nobn: p.data += -lr * p.grad

# Değerlendirme
emb_val = C_nobn[Xdev]
h_val = torch.tanh(emb_val.view(emb_val.shape[0], -1) @ W1_nobn + b1_nobn)
logits_val = h_val @ W2_nobn + b2_nobn
loss_val_nobn = F.cross_entropy(logits_val, Ydev)
print("BatchNormsuz val score:", loss_val_nobn.item())

BatchNormsuz val score: 2.1456737518310547
